## Regression Returns

Description:     
In this approach, we treat the next-bar (or multi-bar) return as a continuous variable and use a regression model (e.g., RandomForestRegressor) to predict it. Positive predicted returns imply a potential buy signal, negative imply a sell, and near-zero might mean no trade. This method captures magnitude of price movement rather than just direction.

#### 📌 Important Note:
This notebook contains *interactive charts generated using Vectorbt.  
GitHub does not display interactive Plotly charts, so the graphs will not be visible here.  

✅ To view the charts, please download this notebook and run it on your local machine.  
Make sure you have Vectorbt and its dependencies installed to regenerate the visualizations.


## Part 1: Data & Feature Engineering

**Objective:**  
Load raw price data (MetaTrader 5 or CSV) and transform it into a feature-rich dataset.

**Tasks:**
- Fetch historical bars  
- Apply `ta.add_all_ta_features` or custom features  
- (Optionally) create specific labels (multi-bar, double-barrier, regime, etc.)  
- Clean/prepare the final feature matrix **X** and target **y**  

In [1]:
import os
from pathlib import Path

# Assuming the notebook is inside the notebooks folder, set the project root as one level up.
project_root = Path.cwd().parent
os.chdir(project_root)


import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import MetaTrader5 as mt5
import vectorbt as vbt


# Our modules
from data.data_loader import get_data_mt5
from features.feature_engineering import add_all_ta_features
from models.model_training import (
    select_features_rf_reg,
    walk_forward_splits
)
# Updated simulate_trading that handles cost
from backtests.simple_backtest import simulate_trading, calculate_sharpe_ratio

from features.labeling_schemes import calculate_future_returns

# Sklearn / Models
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor
)
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor



###########################################################
# 1) DATA LOADING & FEATURE ENGINEERING
###########################################################
if not mt5.initialize():
    print("Failed to initialize MT5")
else:
    data = get_data_mt5(symbol="BTCUSD", timeframe=mt5.TIMEFRAME_H4, n_bars=1000)
    mt5.shutdown()

df = add_all_ta_features(data)
df = calculate_future_returns(df).dropna(subset=["future_returns"])

# Prepare X, y
X = df.drop(columns=["future_returns"])
y = df["future_returns"]

###########################################################
# 2) WALK-FORWARD SPLITS
###########################################################
folds = walk_forward_splits(X, y, n_splits=3)
print(f"Number of folds created: {len(folds)}")

###########################################################
# 3) DEFINE MODELS
###########################################################
models = {
    "GradientBoostingRegressor": GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42),
    "RandomForestRegressor": RandomForestRegressor(n_estimators=100, random_state=42),
    "LinearRegression": LinearRegression(),
    "SVR": SVR(C=1.0, epsilon=0.2),
    "KNeighborsRegressor": KNeighborsRegressor(n_neighbors=5),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=0.1),
    "ElasticNet": ElasticNet(alpha=0.1, l1_ratio=0.5),
    "DecisionTreeRegressor": DecisionTreeRegressor(max_depth=5),
    "XGBRegressor": XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42),
    "LGBMRegressor": LGBMRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
}

###########################################################
# 4) LOOP OVER FOLDS + SIMPLE BACKTEST (with threshold & cost)
###########################################################
from sklearn.metrics import mean_squared_error

fold_results = {}
threshold = 0.0005  # Only trade if predicted next-bar return > +0.0005 or < -0.0005
cost = 0.0002       # 0.02% transaction cost each position change

for fold_i, (X_train_fold, y_train_fold, X_test_fold, y_test_fold) in enumerate(folds, start=1):
    print(f"\n===== Fold {fold_i} =====")

    # Feature selection
    rf_for_fs = RandomForestRegressor(n_estimators=100, random_state=42)
    X_train_sel, selected_idx = select_features_rf_reg(
        X_train_fold, y_train_fold, estimator=rf_for_fs, max_features=20
    )

    # Convert integer positions to actual column names
    feats = X_train_fold.columns[selected_idx]
    print(f"Selected features for Fold {fold_i}: {feats.tolist()}")

    X_test_sel = X_test_fold[feats]

    # Scale
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_sel)
    X_test_scaled = scaler.transform(X_test_sel)

    # We'll store MSE and backtest results for each model
    fold_results[fold_i] = {}

    for model_name, model in models.items():
        # 4.1 Train
        model.fit(X_train_scaled, y_train_fold)
        preds = model.predict(X_test_scaled)

        # 4.2 Evaluate MSE
        mse = mean_squared_error(y_test_fold, preds)

        # 4.3 Convert predictions => signals with threshold
        signals = np.where(preds > threshold, 1, np.where(preds < -threshold, -1, 0))

        # 4.4 Align the test portion of df with the same indices
        df_test_fold = df.loc[X_test_fold.index].copy()

        # 4.5 Run the simple backtest with cost
        daily_returns, total_return = simulate_trading(signals, df_test_fold, cost=cost)
        sr = calculate_sharpe_ratio(np.array(daily_returns))

        fold_results[fold_i][model_name] = {
            "MSE": mse,
            "TotalReturn": total_return,
            "Sharpe": sr
        }

    # End of model loop

# End of fold loop

###########################################################
# 5) PRINT RESULTS
###########################################################
for fold_i, model_dict in fold_results.items():
    print(f"\n=== Fold {fold_i} Results ===")
    for model_name, stats in model_dict.items():
        mse = stats["MSE"]
        ret = stats["TotalReturn"]
        sr = stats["Sharpe"]
        print(f"{model_name}: MSE={mse:.2e}, Return={ret:.2f}%, Sharpe={sr:.2f}")


Number of folds created: 3

===== Fold 1 =====
Selected features for Fold 1: ['trend_dpo', 'volume_obv', 'trend_kst_diff', 'volume_adi', 'tick_volume', 'momentum_pvo_hist', 'trend_mass_index', 'volatility_kcw', 'volatility_bbw', 'trend_adx', 'volume_nvi', 'volume_em', 'momentum_ao', 'trend_stc', 'volatility_dcw', 'trend_adx_neg', 'trend_vortex_ind_pos', 'momentum_stoch_signal', 'volatility_atr', 'momentum_roc']


  File "c:\Users\moham\miniconda3\envs\ml\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "c:\Users\moham\miniconda3\envs\ml\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\moham\miniconda3\envs\ml\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "c:\Users\moham\miniconda3\envs\ml\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000251 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1650
[LightGBM] [Info] Number of data points in the train set: 249, number of used features: 20
[LightGBM] [Info] Start training from score 0.000954
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, b

## Part 2: Model Training & Hyperparameter Tuning

**Objective:**  
Train an ML model (e.g., RandomForest, XGBoost) on the engineered features to predict the chosen labels.

**Tasks:**
- Perform time-based or walk-forward splits  
- Select top features if desired (e.g., using RandomForest feature importance)  
- Use `RandomizedSearchCV` or `GridSearchCV` to find optimal hyperparameters  
- Save the best model pipeline (e.g., `best_rf_pipeline.pkl`) 

In [1]:
# Code 2: Hyperparameter Tuning for Chosen Model

import os
from pathlib import Path

# Assuming the notebook is inside the notebooks folder, set the project root as one level up.
project_root = Path.cwd().parent
os.chdir(project_root)

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import MetaTrader5 as mt5
import joblib

# Sklearn / Models
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import make_scorer, mean_squared_error

# Optional: If you want a pipeline
from sklearn.pipeline import Pipeline

# Your modules
from data.data_loader import get_data_mt5
from features.feature_engineering import add_all_ta_features
from features.labeling_schemes import calculate_future_returns

###########################################################
# 1) DATA LOADING & FEATURE ENGINEERING
###########################################################
# In Code 1, you discovered that RandomForestRegressor generally outperforms.
# Now we do a narrower hyperparam tuning on the older portion of the data.

if not mt5.initialize():
    print("Failed to initialize MT5")
else:
    data = get_data_mt5(symbol="BTCUSD", timeframe=mt5.TIMEFRAME_H4, n_bars=2000)
    mt5.shutdown()

df = add_all_ta_features(data)
df = calculate_future_returns(df).dropna(subset=["future_returns"])

X_full = df.drop(columns=["future_returns"])
y_full = df["future_returns"]

# Sort df if needed to ensure chronological order
# df = df.sort_index()  # or df = df.sort_values('time') if you have that column
# Then X_full and y_full should match that order

###########################################################
# 2) DEFINE YOUR TRAIN PORTION
###########################################################
# For example, let's use the first 80% for hyperparam tuning
split_idx = int(len(X_full)*0.8)
X_tune = X_full.iloc[:split_idx].copy()
y_tune = y_full.iloc[:split_idx].copy()

print(f"Tuning portion size: {len(X_tune)}")

###########################################################
# 3) TIME-BASED CV (TimeSeriesSplit)
###########################################################
tscv = TimeSeriesSplit(n_splits=3)

# We'll define a negative MSE scorer, because scikit-learn tries to maximize the score
def neg_mse_scorer(estimator, X, y):
    preds = estimator.predict(X)
    return -mean_squared_error(y, preds)

scorer = make_scorer(mean_squared_error, greater_is_better=False)

###########################################################
# 4) BUILD A PIPELINE (optional) OR JUST USE RF
###########################################################
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("rf", RandomForestRegressor(random_state=42))
])

###########################################################
# 5) DEFINE PARAM DISTRIBUTIONS FOR RandomForest
###########################################################
param_distributions = {
    "rf__n_estimators": [100, 200, 300],
    "rf__max_depth": [None, 5, 10, 15],
    "rf__min_samples_split": [2, 5, 10],
    "rf__max_features": ["auto", "sqrt", 0.5]
}

###########################################################
# 6) SET UP RandomizedSearchCV
###########################################################
from sklearn.model_selection import RandomizedSearchCV

random_search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_distributions,
    n_iter=10,             # how many random combos
    scoring=neg_mse_scorer,  # or you can use 'neg_mean_squared_error'
    cv=tscv,               # time-based folds
    random_state=42,
    n_jobs=-1,
    verbose=2
)

###########################################################
# 7) FIT ON TUNING PORTION
###########################################################
random_search.fit(X_tune, y_tune)

print("Best params:", random_search.best_params_)
print("Best neg MSE:", random_search.best_score_)

best_estimator = random_search.best_estimator_

###########################################################
# 8) SAVE THE BEST ESTIMATOR
###########################################################
joblib.dump(best_estimator, "models/saved_models/best_rf_pipeline.pkl")
print("Saved best estimator to 'best_rf_pipeline.pkl'")


Tuning portion size: 1599
Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best params: {'rf__n_estimators': 100, 'rf__min_samples_split': 2, 'rf__max_features': 0.5, 'rf__max_depth': 5}
Best neg MSE: -0.0001469375124074844
Saved best estimator to 'best_rf_pipeline.pkl'


In [3]:
import os
print(os.getcwd())


c:\Users\moham\OneDrive


## Part 3: Backtesting & Performance Evaluation

**Objective:**  
Evaluate how well the trained model performs on unseen data, simulating real trades.

**Tasks:**
- Use walk-forward or expanding splits to mimic “live” conditions  
- Convert model predictions to signals ([-1, 0, +1] or buy/sell/hold)  
- Run a simple backtest script or VectorBT for performance metrics  
- Calculate returns, Sharpe ratio, drawdowns, confusion matrix, etc.  
- Visualize results (equity curve, trades, etc.) to judge strategy viability  

In [1]:
# THIRD CODE: Final Expanding Walk-Forward with VectorBT
# using the best pipeline loaded from best_rf_pipeline.pkl

import os
from pathlib import Path

# Assuming the notebook is inside the notebooks folder, set the project root as one level up.
project_root = Path.cwd().parent
os.chdir(project_root)

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import MetaTrader5 as mt5
import vectorbt as vbt
import joblib

# Our modules
from data.data_loader import get_data_mt5
from features.feature_engineering import add_all_ta_features
from features.labeling_schemes import calculate_future_returns

# Sklearn
from sklearn.metrics import mean_squared_error

###########################################################
# 1) DATA LOADING & FEATURE ENGINEERING
###########################################################
if not mt5.initialize():
    print("Failed to initialize MT5")
else:
    data = get_data_mt5(symbol="BTCUSD", timeframe=mt5.TIMEFRAME_H4, n_bars=2000)
    mt5.shutdown()

df = add_all_ta_features(data)
df = calculate_future_returns(df).dropna(subset=["future_returns"])

# If df's index is date/time, ensure it's sorted chronologically if needed:
# df = df.sort_index()

X = df.drop(columns=["future_returns"])
y = df["future_returns"]

###########################################################
# 2) DEFINING EXPANDING SPLITS
###########################################################
def expanding_walk_forward_splits(X, y, n_splits=3):
    """
    Creates expanding walk-forward folds. For each fold i:
      - Train: [0 : fold_i]
      - Test:  [fold_i : fold_{i+1}]
    The last fold extends to the end.
    """
    n = len(X)
    fold_size = n // (n_splits + 1)
    splits = []
    
    for i in range(n_splits):
        train_end = (i+1) * fold_size
        if i == n_splits - 1:
            test_end = n
        else:
            test_end = (i+2) * fold_size
            if test_end > n:
                test_end = n
        
        X_train_fold = X.iloc[:train_end]
        y_train_fold = y.iloc[:train_end]
        
        X_test_fold = X.iloc[train_end:test_end]
        y_test_fold = y.iloc[train_end:test_end]
        
        splits.append((X_train_fold, y_train_fold, X_test_fold, y_test_fold))
    return splits

folds = expanding_walk_forward_splits(X, y, n_splits=3)
print(f"Number of folds: {len(folds)}")

###########################################################
# 3) LOAD BEST PIPELINE & RUN EXPANDING WALK-FORWARD
###########################################################
# This pipeline was saved in the second code (hyperparam tuning step)
best_pipeline = joblib.load("models/saved_models/best_rf_pipeline.pkl")
print("Loaded best pipeline from 'best_rf_pipeline.pkl'")

threshold = 0.0005  # min predicted return for a trade
fees = 0.0002       # 0.02% transaction cost per trade

fold_results = {}

for i, (X_train_fold, y_train_fold, X_test_fold, y_test_fold) in enumerate(folds, start=1):
    print(f"\n=== Fold {i} ===")
    print(f"Train size: {len(X_train_fold)}, Test size: {len(X_test_fold)}")
    
    # 3.1 Fit the pipeline on this fold's train portion
    # The pipeline includes: [("scaler", StandardScaler()), ("rf", RandomForestRegressor with best hyperparams)]
    best_pipeline.fit(X_train_fold, y_train_fold)
    
    # 3.2 Predict on the test portion
    preds = best_pipeline.predict(X_test_fold)
    mse = mean_squared_error(y_test_fold, preds)
    
    # 3.3 Convert predictions => signals with threshold
    signals = np.where(preds > threshold, 1, np.where(preds < -threshold, -1, 0))
    
    # 3.4 Vectorbt backtest on the test portion
    df_test_fold = df.loc[X_test_fold.index].copy()  # label-based indexing
    close_prices = df_test_fold["close"]
    
    # pad signals if needed
    if len(signals) < len(close_prices):
        signals = np.append(signals, [0]*(len(close_prices)-len(signals)))
    
    signals_s = pd.Series(signals, index=close_prices.index)
    
    pf = vbt.Portfolio.from_signals(
        close_prices,
        entries=signals_s > 0,
        exits=signals_s < 0,
        init_cash=10000,
        freq='4H',
        fees=fees
    )
    
    total_return = pf.total_return()
    sharpe_ratio = pf.sharpe_ratio()
    
    print(f"Fold {i} MSE={mse:.2e}, Return={total_return:.2f}%, Sharpe={sharpe_ratio:.2f}")
    
    # Full stats
    print("\nVectorbt stats for Fold", i)
    print(pf.stats())

    # Optional: Plot
    fig = pf.plot()
    fig.show()
    
    # store results
    fold_results[i] = {
        "MSE": mse,
        "TotalReturn": total_return,
        "Sharpe": sharpe_ratio
    }

###########################################################
# 4) PRINT FOLD RESULTS
###########################################################
for i, stats in fold_results.items():
    print(f"\nFold {i} => MSE={stats['MSE']:.2e}, Return={stats['TotalReturn']:.2f}%, Sharpe={stats['Sharpe']:.2f}")


Number of folds: 3
Loaded best pipeline from 'best_rf_pipeline.pkl'

=== Fold 1 ===
Train size: 499, Test size: 499
Fold 1 MSE=1.76e-04, Return=0.01%, Sharpe=0.35

Vectorbt stats for Fold 1
Start                         2024-06-17 08:00:00
End                           2024-09-08 08:00:00
Period                           83 days 04:00:00
Start Value                               10000.0
End Value                            10111.759885
Total Return [%]                         1.117599
Benchmark Return [%]                   -17.666479
Max Gross Exposure [%]                      100.0
Total Fees Paid                         77.335999
Max Drawdown [%]                        18.499058
Max Drawdown Duration            26 days 16:00:00
Total Trades                                   19
Total Closed Trades                            18
Total Open Trades                               1
Open Trade PnL                          77.630515
Win Rate [%]                            55.555556
Best Trade


=== Fold 2 ===
Train size: 998, Test size: 499
Fold 2 MSE=1.06e-04, Return=0.07%, Sharpe=3.58

Vectorbt stats for Fold 2
Start                               2024-09-08 12:00:00
End                                 2024-11-30 12:00:00
Period                                 83 days 04:00:00
Start Value                                     10000.0
End Value                                  10674.364639
Total Return [%]                               6.743646
Benchmark Return [%]                          78.118375
Max Gross Exposure [%]                            100.0
Total Fees Paid                               16.762278
Max Drawdown [%]                               2.193692
Max Drawdown Duration                   2 days 16:00:00
Total Trades                                          4
Total Closed Trades                                   4
Total Open Trades                                     0
Open Trade PnL                                      0.0
Win Rate [%]                          


=== Fold 3 ===
Train size: 1497, Test size: 502
Fold 3 MSE=1.24e-04, Return=0.08%, Sharpe=1.41

Vectorbt stats for Fold 3
Start                         2024-11-30 16:00:00
End                           2025-02-22 04:00:00
Period                           83 days 16:00:00
Start Value                               10000.0
End Value                            10846.367725
Total Return [%]                         8.463677
Benchmark Return [%]                    -0.369181
Max Gross Exposure [%]                      100.0
Total Fees Paid                         90.854135
Max Drawdown [%]                         7.703264
Max Drawdown Duration            39 days 16:00:00
Total Trades                                   23
Total Closed Trades                            22
Total Open Trades                               1
Open Trade PnL                         274.567578
Win Rate [%]                            54.545455
Best Trade [%]                            9.55754
Worst Trade [%]            


Fold 1 => MSE=1.76e-04, Return=0.01%, Sharpe=0.35

Fold 2 => MSE=1.06e-04, Return=0.07%, Sharpe=3.58

Fold 3 => MSE=1.24e-04, Return=0.08%, Sharpe=1.41
